
# Reporte contrastivo — E1 MRI + impacto E2 (IL-CLDM)

Comparación de la **wave contrastiva E1 MRI** (baseline + 2×2 SupCon/Xmodal × z18k/h64) para justificar **`xmodal_h64`** como guía, y su impacto en **E2 difusión ROI** (`data_rois/`).

**E1 (Parte I):** los cinco experimentos ROI con contrastivo o baseline recon:
`mri_rois`, `supcon_z18k`, `supcon_h64`, `xmodal_z18k`, `xmodal_h64`. Objetivo: evidencia de que `xmodal_h64` ofrece la mejor guía (clasificación, convergencia contrastiva, retrieval xmodal, separación visual).

**E2 (Parte II):** solo baseline `calib_roi3ch_uni_rois` y variantes **xm64** con encoder `xmodal_h64` (seleccionado en Parte I):
- `xm64_z18` — conditioner bottleneck z18k, difusión z18k
- `xm64_h64` — conditioner + difusión h64 (64ch)
- `xm64_h64_gap1` — conditioner h64, GAP→1ch, difusión 1ch

**Métrica primaria:** balanced accuracy (embeddings E1 directos; latentes sintéticos E2). PSNR/SSIM secundarios en reconstrucción SPECT.

**Export:** `result/stage2_contrastive_wave_summary/`

Reporte E2 general (wave roi3ch sin contrastivo): `code/stage2_experiments_report.ipynb`.



## 0. Configuración


In [ ]:
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:

import sys
import importlib
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, here.parent):
        if (candidate / "result").is_dir() and (candidate / "code" / "contrastive_report_utils.py").is_file():
            return candidate
    raise RuntimeError("Ejecuta desde IL-CLDM/ o IL-CLDM/code/.")


ROOT = find_repo_root()
os.chdir(ROOT)
root_str = str(ROOT)
if root_str in sys.path:
    sys.path.remove(root_str)
sys.path.insert(0, root_str)

for _mod in list(sys.modules):
    if _mod == "code" or _mod.startswith("code.") or _mod.endswith(
        ("contrastive_report_utils", "stage2_report_utils", "diffusion_features", "volume_shapes", "embedding_analysis")
    ):
        del sys.modules[_mod]
if "code" in sys.modules and not hasattr(sys.modules.get("code"), "__path__"):
    del sys.modules["code"]

import code.contrastive_report_utils as contrastive_report_utils
import code.stage2_report_utils as stage2_report_utils
importlib.reload(stage2_report_utils)
importlib.reload(contrastive_report_utils)

from code.contrastive_report_utils import (
    E1_E2_LINK_ROWS,
    E1_EXPERIMENTS,
    E1_ORDER,
    E2_ORDER,
    EXPORT_DIR_NAME,
    analysis_dir,
    available_e1_experiments,
    E1_SELECTED_KEY,
    build_e1_leaderboard,
    build_e1_e2_summary_table,
    e2_comparison_from_registry,
    export_dir,
    load_crossmodal_retrieval,
    load_e1_internal_test_embeddings,
    load_phase1_classification,
    load_phase1_projection_csv,
    load_phase1_run_config,
    load_phase1_validation_summary,
    load_e2_classification,
    load_e2_reconstruction,
    best_classifier_row,
    validate_e1_artifacts,
    validate_e2_artifacts,
)
from code.stage2_report_utils import (
    LABEL_NAMES,
    load_synthetic_volume_for,
    load_test_latent_matrix_for,
    load_training_frames_for,
    pick_stratified_subjects,
    repo_root,
    training_checkpoint_summary,
    subset_by_modes,
)
from viz_pairs import display_range

E1_COMPARISON = available_e1_experiments()
E1_KEYS = [k for k in E1_ORDER if k in E1_COMPARISON]
if len(E1_KEYS) < len(E1_ORDER):
    missing = [k for k in E1_ORDER if k not in E1_COMPARISON]
    raise RuntimeError(f"Faltan resultados E1: {missing}")

E2_COMPARISON = e2_comparison_from_registry()
E2_MODES = [m for m in E2_ORDER if m in E2_COMPARISON]
if len(E2_MODES) < len(E2_ORDER):
    missing = [m for m in E2_ORDER if m not in E2_COMPARISON]
    raise RuntimeError(f"Faltan resultados E2: {missing}")

E1_LABELS = {k: E1_COMPARISON[k]["label"] for k in E1_KEYS}
E1_SHORT = {k: E1_COMPARISON[k]["short_label"] for k in E1_KEYS}
E2_LABELS = {m: E2_COMPARISON[m]["label"] for m in E2_MODES}
E2_SHORT = {m: E2_COMPARISON[m]["short_label"] for m in E2_MODES}

_PALETTE = ["#4C72B0", "#C44E52", "#55A868", "#DD8452", "#8172B2"]
E1_COLORS = {k: _PALETTE[i % len(_PALETTE)] for i, k in enumerate(E1_KEYS)}
E2_COLORS = {m: _PALETTE[i % len(_PALETTE)] for i, m in enumerate(E2_MODES)}

REPORT_EXPORT_DIR = export_dir()
DATA_ROIS = repo_root() / "data_rois"

sns.set_theme(style="whitegrid", context="notebook")
%matplotlib inline

print(f"Repo: {repo_root()}")
print(f"E1 experimentos: {E1_KEYS}")
print(f"E2 experimentos: {E2_MODES}")
print(f"Export: {REPORT_EXPORT_DIR}")
for w in validate_e1_artifacts(E1_KEYS):
    print(f"WARNING E1: {w}")
for w in validate_e2_artifacts(E2_COMPARISON):
    print(f"WARNING E2: {w}")





## 0b. Mapa E1 → E2 (encoder y espacio de difusión)

Los embeddings E1 analizados en Parte I usan el **mismo checkpoint MRI** y el **mismo espacio de features** que la condición UNet en E2 (mapa espacial antes de flatten).



In [ ]:

link_df = pd.DataFrame(E1_E2_LINK_ROWS)
display(link_df)




## Parte I — E1 MRI contrastivo (wave completa)

Comparación de **todos** los experimentos E1 MRI con contrastivo (más baseline recon). La selección de `xmodal_h64` para E2 se documenta en §3b.

### 1. Configuración E1 (`run_config.json`)



In [ ]:

e1_config_rows = []
for key in E1_KEYS:
    cfg = E1_COMPARISON[key]
    rc = load_phase1_run_config(cfg["exp"]) or {}
    e1_config_rows.append({
        "e1_key": key,
        "short_label": cfg["short_label"],
        "exp": cfg["exp"],
        "training_mode": rc.get("training_mode", cfg.get("training_mode")),
        "contrastive_features": rc.get("contrastive_features", cfg.get("contrastive_features")),
        "contrastive_temp": rc.get("contrastive_temp"),
        "supcon_lambda": rc.get("supcon_lambda"),
        "xmodal_lambda": rc.get("xmodal_lambda"),
        "projector": rc.get("projector"),
        "checkpoint_selection": rc.get("checkpoint_selection"),
        "batch_size": rc.get("batch_size"),
    })
e1_config_df = pd.DataFrame(e1_config_rows)
e1_config_df




### 2. Convergencia E1

Fuente: `phase1_mri/reconstruction/validation_summary.csv` — `val_contrastive_loss` (si aplica) y PSNR/SSIM de reconstrucción MRI.



In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for key in E1_KEYS:
    cfg = E1_COMPARISON[key]
    val = load_phase1_validation_summary(cfg["exp"])
    if val.empty:
        continue
    short = E1_SHORT[key]
    color = E1_COLORS[key]
    if "val_contrastive_loss" in val.columns and val["val_contrastive_loss"].notna().any():
        axes[0].plot(val["Epoch"], val["val_contrastive_loss"], label=short, color=color, linewidth=1.5)
    axes[1].plot(val["Epoch"], val["PSNR"], label=short, color=color, linewidth=1.5)
axes[0].set_title("val_contrastive_loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[1].set_title("Reconstrucción MRI — PSNR validación")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()




### 3. Clasificación E1 (embeddings directos del encoder)

Protocolo: split train original → 80/20 interno; clasificadores sobre **flatten** del mapa (sin L2 explícito en eval; contrastivo usa L2 en entrenamiento).



In [ ]:

e1_cls_frames = []
for key in E1_KEYS:
    cfg = E1_COMPARISON[key]
    df = load_phase1_classification(cfg["exp"], cfg["analysis_name"])
    if df.empty:
        continue
    df = df.copy()
    df["e1_key"] = key
    df["e1_short"] = E1_SHORT[key]
    df["e1_label"] = E1_LABELS[key]
    df["feature_space"] = cfg["contrastive_features"]
    e1_cls_frames.append(df)

# xmodal_h64 bottleneck eval (guía xm64_z18)
if "e1_xmodal_h64" in E1_KEYS:
    cfg = E1_COMPARISON["e1_xmodal_h64"]
    df_z = load_phase1_classification(cfg["exp"], "embedding_analysis_z18k")
    if not df_z.empty:
        df_z = df_z.copy()
        df_z["e1_key"] = "e1_xmodal_h64_z18k"
        df_z["e1_short"] = "xmodal_h64_z18k"
        df_z["e1_label"] = "xmodal_h64 eval z18k bottleneck"
        df_z["feature_space"] = "z18k"
        e1_cls_frames.append(df_z)

e1_cls_all = pd.concat(e1_cls_frames, ignore_index=True)
metric_cols = [c for c in ("accuracy", "balanced_accuracy", "precision", "recall", "f1", "roc_auc") if c in e1_cls_all.columns]
display(e1_cls_all.pivot_table(index="model", columns="e1_short", values=metric_cols).round(3))

fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(
    data=e1_cls_all,
    x="model",
    y="balanced_accuracy",
    hue="e1_short",
    ax=ax,
)
ax.set_title("E1 — balanced accuracy (internal 80/20 test)")
ax.tick_params(axis="x", rotation=35)
ax.legend(title="E1", fontsize=8)
plt.tight_layout()
plt.show()

e1_best = (
    e1_cls_all.sort_values(["balanced_accuracy", "roc_auc"], ascending=False)
    .groupby("e1_short", as_index=False)
    .first()
)
e1_best[["e1_short", "feature_space", "model", "balanced_accuracy", "roc_auc", "f1"]].round(3)



### 3b. Evidencia para elegir `xmodal_h64` en E2

Leaderboard E1: clasificación interna 80/20, pérdida contrastiva en validación y retrieval xmodal (val).
`xmodal_h64` destaca en balanced accuracy y retrieval; variantes E2 `xm64_*` usan su checkpoint MRI.


In [ ]:
e1_leaderboard = build_e1_leaderboard(E1_COMPARISON)
display(e1_leaderboard.round(4))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
plot_df = e1_leaderboard.sort_values("balanced_accuracy", ascending=True)
colors = ["#d83b5f" if r["selected_for_e2"] else "#94a3b8" for _, r in plot_df.iterrows()]
axes[0].barh(plot_df["e1_short"], plot_df["balanced_accuracy"], color=colors)
axes[0].set_title("Balanced accuracy (E1 cls)")
axes[0].set_xlim(0, 1)

ctr = plot_df.dropna(subset=["min_val_contrastive_loss"])
if len(ctr):
    axes[1].barh(ctr["e1_short"], ctr["min_val_contrastive_loss"], color="#64748b")
    axes[1].set_title("Min val contrastive loss")
else:
    axes[1].set_visible(False)

ret = plot_df.dropna(subset=["xmodal_top1_retrieval"])
if len(ret):
    axes[2].barh(ret["e1_short"], ret["xmodal_top1_retrieval"], color="#3b6fd8")
    axes[2].set_title("Xmodal top-1 retrieval (val)")
else:
    axes[2].set_visible(False)

plt.tight_layout()
plt.show()


### 4. Visualización embeddings E1 (PCA / t-SNE / UMAP)

Proyecciones precomputadas en `embedding_analysis/visualizations/` (internal test split).
Color: **control** azul (`#3b6fd8`), **parkinson** rojo (`#d83b5f`).


In [ ]:
PROJ_METHODS = ("pca", "tsne", "umap")
CLASS_PALETTE = {"control": "#3b6fd8", "parkinson": "#d83b5f"}
CLASS_HUE_ORDER = ("control", "parkinson")


def _class_column(df: pd.DataFrame) -> str | None:
    for col in ("class_name", "class"):
        if col in df.columns:
            return col
    if "label" in df.columns:
        df["class_name"] = df["label"].map({0: "control", 1: "parkinson", 0.0: "control", 1.0: "parkinson"})
        return "class_name"
    return None


def plot_e1_embedding_projections(exp_name: str, analysis: str, title: str) -> None:
    fig, axes = plt.subplots(1, len(PROJ_METHODS), figsize=(14, 4))
    for ax, method in zip(axes, PROJ_METHODS):
        df2 = load_phase1_projection_csv(exp_name, method, 2, analysis)
        if df2 is None:
            ax.set_visible(False)
            continue
        hue_col = _class_column(df2)
        if hue_col is None:
            ax.scatter(df2[f"{method}_1"], df2[f"{method}_2"], s=55, alpha=0.85, color="#888")
        else:
            sns.scatterplot(
                data=df2,
                x=f"{method}_1",
                y=f"{method}_2",
                hue=hue_col,
                hue_order=list(CLASS_HUE_ORDER),
                palette=CLASS_PALETTE,
                ax=ax,
                s=55,
                alpha=0.85,
            )
            ax.legend(title="", fontsize=8)
        ax.set_title(method.upper())
    fig.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()


for key in E1_KEYS:
    cfg = E1_COMPARISON[key]
    plot_e1_embedding_projections(
        cfg["exp"],
        cfg["analysis_name"],
        f"E1 embeddings — {E1_SHORT[key]} ({cfg['contrastive_features']})",
    )

# z18k bottleneck from xmodal_h64 encoder
if "e1_xmodal_h64" in E1_KEYS:
    cfg = E1_COMPARISON["e1_xmodal_h64"]
    plot_e1_embedding_projections(
        cfg["exp"],
        "embedding_analysis_z18k",
        "E1 embeddings — xmodal_h64 encoder, z18k bottleneck eval",
    )




### 5. Retrieval cross-modal (xmodal_h64)

Fuente: `crossmodal_retrieval.json` en validación (MRI→SPECT, mismo sujeto).



In [ ]:

import json

if "e1_xmodal_h64" in E1_KEYS:
    cfg = E1_COMPARISON["e1_xmodal_h64"]
    retrieval = load_crossmodal_retrieval(cfg["exp"], cfg["analysis_name"])
    if retrieval:
        print(json.dumps(retrieval, indent=2))
        fig, ax = plt.subplots(figsize=(4, 3))
        ax.bar(["top1", "chance"], [retrieval.get("top1_retrieval", 0), retrieval.get("random_chance", 0)], color=["#C44E52", "#aaa"])
        ax.set_ylim(0, 1)
        ax.set_title("xmodal_h64 — top-1 retrieval (val)")
        plt.tight_layout()
        plt.show()
else:
    print("xmodal_h64 no disponible")




## Parte II — Impacto en E2 (difusión ROI)

### 6. Convergencia entrenamiento E2



In [ ]:

from matplotlib.ticker import MaxNLocator

e2_training = {m: load_training_frames_for(E2_COMPARISON[m]["exp"]) for m in E2_MODES}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for mode in E2_MODES:
    frames = e2_training[mode]
    loss = frames.get("loss_curve")
    if loss is not None and "MSE_loss" in loss.columns:
        axes[0].plot(loss["Epoch"], loss["MSE_loss"], label=E2_SHORT[mode], color=E2_COLORS[mode], linewidth=1.2)
    val_lat = frames.get("validation_latent")
    if val_lat is not None and "latent_MSE" in val_lat.columns:
        axes[1].plot(val_lat["Epoch"], val_lat["latent_MSE"], label=E2_SHORT[mode], color=E2_COLORS[mode], linewidth=1.2)
axes[0].set_title("Train MSE (difusión)")
axes[0].set_xlabel("Epoch")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)
axes[1].set_title("Validación latente MSE")
axes[1].set_xlabel("Epoch")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

e2_ckpt_rows = []
for mode in E2_MODES:
    cfg = E2_COMPARISON[mode]
    frames = e2_training[mode]
    criterion = cfg.get("early_stop", "image")
    summary = training_checkpoint_summary(
        frames["validation"],
        frames["test"],
        criterion=criterion if criterion in {"image", "latent"} else "image",
        validation_latent=frames.get("validation_latent"),
        test_latent=frames.get("test_latent"),
    )
    summary["mode"] = mode
    summary["mode_short"] = E2_SHORT[mode]
    e2_ckpt_rows.append(summary)
e2_ckpt_df = pd.DataFrame(e2_ckpt_rows)
e2_ckpt_df




### 7. Reconstrucción SPECT en test (inferencia)



In [ ]:

e2_recon_by_mode = {m: load_e2_reconstruction(m, E2_COMPARISON) for m in E2_MODES}
e2_recon_all = pd.concat(e2_recon_by_mode.values(), ignore_index=True)
metric_cols = ["PSNR", "SSIM"]
if "PSNR_roi" in e2_recon_all.columns:
    metric_cols.extend(["PSNR_roi", "SSIM_roi"])

display(
    e2_recon_all.groupby("mode_short")[metric_cols].agg(["mean", "std"]).round(3)
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, col in zip(axes, ("PSNR", "SSIM")):
    sns.boxplot(
        data=e2_recon_all,
        x="mode_short",
        y=col,
        order=[E2_SHORT[m] for m in E2_MODES],
        palette=[E2_COLORS[m] for m in E2_MODES],
        ax=ax,
    )
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=25)
plt.suptitle("Reconstrucción SPECT test — variantes xm64 vs baseline ROI", y=1.02)
plt.tight_layout()
plt.show()




### 8. Visualizaciones SPECT comparables (mismos sujetos test)



In [ ]:

from dataset import nifti_to_numpy, resolve_nifti_path

def load_gt_spect_roi(subject_id: str) -> np.ndarray:
    root = DATA_ROIS / "whole_Abeta"
    return nifti_to_numpy(str(resolve_nifti_path(str(root), subject_id)), zscore=True)

selected = pick_stratified_subjects(e2_recon_by_mode)

def plot_spect_grid(selected_df: pd.DataFrame, modes: list[str], title: str) -> None:
    col_titles = ["GT SPECT"] + [E2_SHORT[m] for m in modes]
    n_cols = len(col_titles)
    n_rows = len(selected_df)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.2 * n_cols, 3.0 * n_rows))
    if n_rows == 1:
        axes = np.expand_dims(axes, axis=0)
    for row_idx, (_, row) in enumerate(selected_df.iterrows()):
        sid = str(row["ID"])
        gt = load_gt_spect_roi(sid)
        z = gt.shape[2] // 2
        gt_lim = display_range(gt, symmetric=False, percentiles=(5.0, 99.5))
        volumes = [gt]
        for mode in modes:
            cfg = E2_COMPARISON[mode]
            volumes.append(load_synthetic_volume_for(cfg["exp"], cfg["run_id"], sid))
        for col_idx, (vol, col_title) in enumerate(zip(volumes, col_titles)):
            ax = axes[row_idx, col_idx]
            lim = gt_lim if col_idx == 0 else display_range(vol, symmetric=False, percentiles=(5.0, 99.5))
            ax.imshow(vol[:, :, z].T, cmap="hot", origin="lower", vmin=lim[0], vmax=lim[1])
            if row_idx == 0:
                ax.set_title(col_title, fontsize=9)
            ax.axis("off")
            if col_idx == 0:
                ax.set_ylabel(f"{sid}\n{row['label_name']}\n{row['tier']}", fontsize=8)
    fig.suptitle(title, y=1.01)
    plt.tight_layout()
    plt.show()

plot_spect_grid(selected, E2_MODES, "SPECT sintético test — ROI patch (data_rois)")
selected[["ID", "label_name", "tier", "psnr_mean"]]




### 9. Clasificación E2 (latentes sintéticos DDIM, test)



In [ ]:

e2_cls_frames = [load_e2_classification(m, E2_COMPARISON) for m in E2_MODES]
e2_cls_all = pd.concat(e2_cls_frames, ignore_index=True)
metric_cols = [c for c in ("accuracy", "balanced_accuracy", "precision", "recall", "f1", "roc_auc") if c in e2_cls_all.columns]
display(e2_cls_all.pivot_table(index="model", columns="mode_short", values=metric_cols).round(3))

fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=e2_cls_all, x="model", y="balanced_accuracy", hue="mode_short", ax=ax)
ax.set_title("E2 — balanced accuracy (latentes sintéticos, test)")
ax.tick_params(axis="x", rotation=35)
ax.legend(title="E2", fontsize=8)
plt.tight_layout()
plt.show()

e2_best = (
    e2_cls_all.sort_values(["balanced_accuracy", "roc_auc"], ascending=False)
    .groupby("mode_short", as_index=False)
    .first()
)
e2_best[["mode_short", "model", "balanced_accuracy", "roc_auc", "f1"]].round(3)




### 10. Visualización latentes E2 test (mejor clasificador por modo)



In [ ]:

from code.embedding_analysis import label_name, projection_coordinates

best_e2_row = e2_cls_all.sort_values(["balanced_accuracy", "roc_auc"], ascending=False).iloc[0]
best_e2_mode = best_e2_row["mode"]
best_e2_model = best_e2_row["model"]
best_e2_cfg = E2_COMPARISON[best_e2_mode]
print(
    f"Mejor E2 global: {E2_SHORT[best_e2_mode]} / {best_e2_model} "
    f"bal.acc={best_e2_row['balanced_accuracy']:.3f}"
)

PROJ_METHODS = ("pca", "tsne", "umap")
for mode in E2_MODES:
    cfg = E2_COMPARISON[mode]
    X_lat, y_lat, ids_lat = load_test_latent_matrix_for(cfg["exp"], cfg["run_id"])
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, method in zip(axes, PROJ_METHODS):
        coords = projection_coordinates(X_lat, method, 2)
        df2 = pd.DataFrame(coords, columns=[f"{method}_1", f"{method}_2"])
        df2["class_name"] = [label_name(y) for y in y_lat]
        sns.scatterplot(
            data=df2,
            x=f"{method}_1",
            y=f"{method}_2",
            hue="class_name",
            hue_order=["control", "parkinson"],
            palette={"control": "#3b6fd8", "parkinson": "#d83b5f"},
            ax=ax,
            s=55,
            alpha=0.85,
        )
        ax.set_title(method.upper())
    fig.suptitle(f"Latentes DDIM test — {E2_SHORT[mode]} (n={len(ids_lat)}, dim={X_lat.shape[1]})", y=1.02)
    plt.tight_layout()
    plt.show()




### 11. Diagnóstico recuperación latente (ruido alto)

Evalúa si el encoder MRI `xmodal_h64` mejora la recuperación del latente SPECT target frente al baseline, desde niveles altos de ruido.

| Familia | Descripción |
|---------|-------------|
| **oracle z_gt** | Techo: latente real SPECT (espacio de difusión del experimento) |
| **t fijo** | `z_gt` → ruido en `t` → un paso `predict_x0` con UNet + guía MRI |
| **DDIM-100** | Muestreo iterativo completo |

Variantes comparadas:
- `calib_roi3ch_uni_rois` — MRI baseline (`mri_rois`)
- `xm64_z18` — conditioner bottleneck z18k del encoder `xmodal_h64`
- `xm64_h64_gap1` — conditioner h64→GAP 1ch + difusión `h64_gap1`

La guía MRI usa `encode_mri_condition` (mismo espacio que en entrenamiento E2). `z_gt` se calcula coherente con `DIFFUSION_FEATURE_SPACE` de cada variante.


**Gráfica adicional:** curva MSE latente + PSNR vs `t` (denoising 1-paso) para cada variante.


In [ ]:
import json
import sys
from contextlib import contextmanager
from pathlib import Path
from tqdm.auto import tqdm

import torch
from torch.utils.data import DataLoader

# El stdlib tiene un módulo `code` (no paquete). Asegura el paquete del repo.
_root = Path.cwd().resolve()
if not (_root / "code" / "diffusion_features.py").is_file():
    for _cand in (_root.parent,):
        if (_cand / "code" / "diffusion_features.py").is_file():
            _root = _cand
            break
_root_str = str(_root)
if _root_str in sys.path:
    sys.path.remove(_root_str)
sys.path.insert(0, _root_str)
if "code" in sys.modules and not hasattr(sys.modules["code"], "__path__"):
    del sys.modules["code"]
for _mod in list(sys.modules):
    if _mod.startswith(("code.diffusion_features", "code.volume_shapes")):
        del sys.modules[_mod]

from code.diffusion_features import encode_mri_condition, resolve_diffusion_feature_space, channel_mean_to_one
from code.volume_shapes import clear_shape_cache
from dataset import TwoDataset
from inference import (
    apply_manifest_to_config,
    load_mri_encoder,
    resolve_mri_encoder_checkpoint,
    resolve_unet_build_config,
    subject_id_from_name,
)
from main import Diffusion, batch_metric_scores, _decode_sampled_latent
from model import AAE, build_unet
from utils import load_checkpoint, seed_torch

import config


DIAG_SAMPLING_MODES = [
    "calib_roi3ch_uni_rois",
    "calib_roi3ch_uni_rois_xm64_z18",
    "calib_roi3ch_uni_rois_xm64_h64_gap1",
]
DIAG_MODE = "frozen"
DIAG_SEED = 42
MAX_SUBJECTS = 15  # None = todos los test (36)
FIXED_T_VALUES = [50, 200, 500, 700, 999]
DDIM_STEPS_LIST = [100]

_CONFIG_KEYS = (
    "SKIP_CROP", "ROI_VOLUME_SHAPE", "whole_MRI", "whole_Abeta", "latent_Abeta",
    "UNET_ROI_CHANNEL", "LATENT_CALIBRATION", "LATENT_STATS_PATH", "LATENT_MEAN", "LATENT_STD",
    "DIFFUSION_FEATURE_SPACE", "DIFFUSION_CHANNELS", "UNET_IN_CHANNELS",
    "DIFFUSION_H64_DECODE_BONUS", "USE_TRAIN_LABEL", "path",
)


@contextmanager
def _e2_profile_context(mode_key: str):
    saved = {k: getattr(config, k, None) for k in _CONFIG_KEYS}
    exp = E2_COMPARISON[mode_key]["exp"]
    manifest = apply_manifest_to_config(exp)
    if manifest and manifest.get("config"):
        cfg = manifest["config"]
        if "DIFFUSION_FEATURE_SPACE" in cfg:
            config.DIFFUSION_FEATURE_SPACE = str(cfg["DIFFUSION_FEATURE_SPACE"])
            config.DIFFUSION_CHANNELS = int(cfg.get("DIFFUSION_CHANNELS", 1))
        if "UNET_IN_CHANNELS" in cfg:
            config.UNET_IN_CHANNELS = int(cfg["UNET_IN_CHANNELS"])
        if "DIFFUSION_H64_DECODE_BONUS" in cfg:
            config.DIFFUSION_H64_DECODE_BONUS = bool(cfg["DIFFUSION_H64_DECODE_BONUS"])
        if "USE_TRAIN_LABEL" in cfg:
            config.USE_TRAIN_LABEL = bool(cfg["USE_TRAIN_LABEL"])
    clear_shape_cache()
    try:
        yield manifest
    finally:
        for k, v in saved.items():
            setattr(config, k, v)
        clear_shape_cache()


def _latent_metrics(z_hat: torch.Tensor, z_gt: torch.Tensor) -> dict:
    a = z_hat.detach().float().cpu().numpy().ravel()
    b = z_gt.detach().float().cpu().numpy().ravel()
    mse = float(np.mean((a - b) ** 2))
    mae = float(np.mean(np.abs(a - b)))
    corr = float(np.corrcoef(a, b)[0, 1]) if a.size > 1 else float("nan")
    return {"latent_mse": mse, "latent_mae": mae, "latent_corr": corr}


def _z_gt_from_spect(aae_model, spect_tensor: torch.Tensor, feat_space: str) -> torch.Tensor:
    if feat_space == "h64":
        _z, h64 = aae_model.encoder(spect_tensor, return_hidden=True)
        return h64
    if feat_space == "h64_gap1":
        _z, h64 = aae_model.encoder(spect_tensor, return_hidden=True)
        return channel_mean_to_one(h64)
    return aae_model.encoder(spect_tensor)


def _image_metrics(aae_model, spect_gt: torch.Tensor, z: torch.Tensor) -> dict:
    with torch.no_grad():
        pred = _decode_sampled_latent(aae_model, z)
        if pred is None:
            return {"PSNR": float("nan"), "SSIM": float("nan")}
    psnr_v, ssim_v = batch_metric_scores(spect_gt.detach().cpu(), pred.detach().cpu())
    return {"PSNR": float(psnr_v), "SSIM": float(ssim_v)}


def load_diag_models(mode_key: str, device):
    import json as _json
    cfg = E2_COMPARISON[mode_key]
    exp = cfg["exp"]
    run_id = cfg["run_id"]
    root = repo_root()
    run_cfg_path = root / "result" / exp / "inference" / run_id / "run_config.json"
    if run_cfg_path.is_file():
        with open(run_cfg_path, encoding="utf-8") as f:
            run_cfg = _json.load(f)
        ckpt_aae = root / run_cfg["checkpoints"]["AAE"]
        ckpt_unet = root / run_cfg["checkpoints"]["Unet"]
        mri_ckpt = root / run_cfg["checkpoints"]["MRI_encoder"]
    else:
        ckpt_aae = root / "result" / exp / "AAE.pth.tar"
        ckpt_unet = root / "result" / exp / "Unet.pth.tar"
        mri_ckpt = resolve_mri_encoder_checkpoint(exp, DIAG_MODE, None)

    use_roi, in_ch = resolve_unet_build_config(exp, ckpt_unet)
    aae = AAE().to(device).eval()
    load_checkpoint(str(ckpt_aae), aae, None, config.learning_rate)
    unet = build_unet(in_channel=in_ch, use_roi_channel=use_roi).to(device).eval()
    load_checkpoint(str(ckpt_unet), unet, None, config.learning_rate)
    mri_encoder, _ = load_mri_encoder(DIAG_MODE, mri_ckpt, device)
    diffusion = Diffusion()
    feat_space = resolve_diffusion_feature_space()
    meta = {
        "feat_space": feat_space,
        "in_channels": in_ch,
        "use_roi": use_roi,
        "latent_mean": float(diffusion.latent_mean),
        "latent_std": float(diffusion.latent_std),
        "use_label": bool(config.USE_TRAIN_LABEL),
    }
    return aae, unet, mri_encoder, diffusion, meta


def _predict_noise(unet_model, x_t, cond, t, label_batch, use_label: bool):
    if use_label:
        return unet_model(x_t, cond, t, label_batch, precomputed_condition=True)
    return unet_model(x_t, cond, t, label=None, precomputed_condition=True)


def _ddim_sample(unet_model, diff_model, cond, label_batch, steps: int, seed: int, use_label: bool):
    seed_torch(seed)

    def _model(x, y, t):
        return _predict_noise(unet_model, x, cond, t, label_batch, use_label)

    return diff_model.sample(_model, cond, sampling_steps=steps, eta=config.ddim_eta)


device = torch.device(config.device)
seed_torch(DIAG_SEED)
print(f"Diagnóstico sampling — variantes: {[E2_SHORT[m] for m in DIAG_SAMPLING_MODES]}")

diag_rows = []
for mode_key in DIAG_SAMPLING_MODES:
    with _e2_profile_context(mode_key):
        data_root = DATA_ROIS
        test_ds = TwoDataset(
            root_MRI=str(data_root / "whole_MRI"),
            root_Abeta=str(data_root / "whole_Abeta"),
            task=str(repo_root() / "data_info" / "test.txt"),
            stage="test",
        )
        if MAX_SUBJECTS is not None:
            test_ds.images = test_ds.images[:MAX_SUBJECTS]
            test_ds.length_dataset = len(test_ds.images)
            test_ds.len = len(test_ds.images)

        loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0)
        aae, unet, mri_encoder, diffusion, meta = load_diag_models(mode_key, device)
        feat_space = meta["feat_space"]
        use_label = meta["use_label"]
        variant_short = E2_SHORT[mode_key]
        variant_label = E2_LABELS[mode_key]
        print(
            f"[{variant_short}] feat={feat_space} in_ch={meta['in_channels']} "
            f"roi={meta['use_roi']} label={use_label} calib_mu={meta['latent_mean']:.3f}"
        )

        for batch_idx, (MRI, Abeta, name, label) in enumerate(
            tqdm(loader, desc=f"sampling diag [{variant_short}]")
        ):
            sid = subject_id_from_name(name)
            MRI_t = torch.as_tensor(np.expand_dims(MRI, 1), dtype=torch.float32, device=device)
            spect_gt = torch.as_tensor(np.expand_dims(Abeta, 1), dtype=torch.float32, device=device)
            label_batch = label.to(device)

            with torch.no_grad():
                z_gt = _z_gt_from_spect(aae, spect_gt, feat_space)
                z_diff = diffusion.normalize_latent(z_gt)
                MRI_cond = encode_mri_condition(mri_encoder, MRI_t, feat_space)

                m_oracle = {**_latent_metrics(z_gt, z_gt), **_image_metrics(aae, spect_gt, z_gt)}
                diag_rows.append({
                    "ID": sid,
                    "variant": mode_key,
                    "variant_short": variant_short,
                    "variant_label": variant_label,
                    "feat_space": feat_space,
                    "method": "oracle_z_gt",
                    "t_or_steps": None,
                    **m_oracle,
                })

                for t_val in FIXED_T_VALUES:
                    t = torch.full((1,), int(t_val), dtype=torch.long, device=device)
                    x_t, _noise = diffusion.noise_images(z_diff, t)
                    eps_pred = _predict_noise(unet, x_t, MRI_cond, t, label_batch, use_label)
                    z_hat_diff = diffusion.predict_x0(x_t, t, eps_pred)
                    z_hat = diffusion.denormalize_latent(z_hat_diff)
                    m = {**_latent_metrics(z_hat, z_gt), **_image_metrics(aae, spect_gt, z_hat)}
                    diag_rows.append({
                        "ID": sid,
                        "variant": mode_key,
                        "variant_short": variant_short,
                        "variant_label": variant_label,
                        "feat_space": feat_space,
                        "method": "fixed_t",
                        "t_or_steps": t_val,
                        **m,
                    })

                for steps in DDIM_STEPS_LIST:
                    z_gen_diff = _ddim_sample(
                        unet, diffusion, MRI_cond, label_batch, steps,
                        seed=DIAG_SEED + batch_idx, use_label=use_label,
                    )
                    z_gen = diffusion.denormalize_latent(z_gen_diff)
                    m = {**_latent_metrics(z_gen, z_gt), **_image_metrics(aae, spect_gt, z_gen)}
                    diag_rows.append({
                        "ID": sid,
                        "variant": mode_key,
                        "variant_short": variant_short,
                        "variant_label": variant_label,
                        "feat_space": feat_space,
                        "method": "ddim",
                        "t_or_steps": steps,
                        **m,
                    })

diag_df = pd.DataFrame(diag_rows)
print(f"Filas: {len(diag_df)} | sujetos: {diag_df['ID'].nunique()} | variantes: {diag_df['variant_short'].unique().tolist()}")
diag_df.head(8)




In [ ]:
def _method_label(row):
    if row["method"] == "oracle_z_gt":
        return "oracle z_gt"
    if row["method"] == "fixed_t":
        return f"t={int(row['t_or_steps'])}"
    return f"DDIM-{int(row['t_or_steps'])}"


diag_df["method_label"] = diag_df.apply(_method_label, axis=1)

summary = (
    diag_df.groupby(
        ["variant_short", "variant_label", "feat_space", "method", "t_or_steps", "method_label"],
        as_index=False,
    )
    .agg(
        latent_mse_mean=("latent_mse", "mean"),
        latent_mse_std=("latent_mse", "std"),
        latent_corr_mean=("latent_corr", "mean"),
        PSNR_mean=("PSNR", "mean"),
        PSNR_std=("PSNR", "std"),
        SSIM_mean=("SSIM", "mean"),
        SSIM_std=("SSIM", "std"),
        n=("latent_mse", "count"),
    )
    .sort_values(["variant_short", "method", "t_or_steps"])
    .round(4)
)
display(summary)

plot_order = (
    ["oracle z_gt"]
    + [f"t={t}" for t in FIXED_T_VALUES]
    + [f"DDIM-{s}" for s in DDIM_STEPS_LIST]
)

for mode_key in DIAG_SAMPLING_MODES:
    variant_short = E2_SHORT[mode_key]
    plot_df = summary[summary["variant_short"] == variant_short].copy()
    plot_df = plot_df[plot_df["method_label"].isin(plot_order)].copy()
    plot_df["method_label"] = pd.Categorical(plot_df["method_label"], categories=plot_order, ordered=True)
    plot_df = plot_df.sort_values("method_label")

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
    x = np.arange(len(plot_df))
    labels = plot_df["method_label"].astype(str)

    axes[0].bar(x, plot_df["latent_mse_mean"], yerr=plot_df["latent_mse_std"], capsize=3, color="steelblue")
    axes[0].set_title("MSE latente vs z_gt")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(labels, rotation=45, ha="right")

    axes[1].bar(x, plot_df["PSNR_mean"], yerr=plot_df["PSNR_std"], capsize=3, color="seagreen")
    axes[1].set_title("PSNR imagen (decode → SPECT)")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(labels, rotation=45, ha="right")

    fig.suptitle(
        f"Diagnóstico recuperación latente — {variant_short} ({E2_LABELS[mode_key]})",
        y=1.02,
    )
    plt.tight_layout()
    plt.show()

    ft = plot_df[plot_df["method"] == "fixed_t"].sort_values("t_or_steps")
    if len(ft):
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.errorbar(
            ft["t_or_steps"],
            ft["latent_mse_mean"],
            yerr=ft["latent_mse_std"],
            fmt="o-",
            color="C0",
            capsize=3,
            label="MSE latente",
        )
        ax2 = ax.twinx()
        ax2.errorbar(
            ft["t_or_steps"],
            ft["PSNR_mean"],
            yerr=ft["PSNR_std"],
            fmt="s--",
            color="green",
            capsize=3,
            label="PSNR imagen",
        )
        ax.set(xlabel="t", ylabel="MSE latente", title=f"1-paso desde z_gt ruidoso — {variant_short}")
        ax2.set_ylabel("PSNR imagen", color="green")
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

# Comparación cruzada en ruido alto
high_noise = summary[summary["method_label"].isin(["t=700", "t=999", "DDIM-100"])].copy()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=high_noise, x="variant_short", y="latent_mse_mean", hue="method_label", ax=axes[0])
axes[0].set_title("MSE latente — t alto / DDIM")
axes[0].tick_params(axis="x", rotation=20)
sns.barplot(data=high_noise, x="variant_short", y="PSNR_mean", hue="method_label", ax=axes[1])
axes[1].set_title("PSNR imagen — t alto / DDIM")
axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

diag_out = REPORT_EXPORT_DIR
diag_out.mkdir(parents=True, exist_ok=True)
diag_df.to_csv(diag_out / "sampling_diagnosis_per_subject.csv", index=False)
summary.to_csv(diag_out / "sampling_diagnosis_summary.csv", index=False)
print(f"Export: {diag_out / 'sampling_diagnosis_*.csv'}")

for variant_short in [E2_SHORT[m] for m in DIAG_SAMPLING_MODES]:
    sub = summary[summary["variant_short"] == variant_short]
    t999 = sub[sub["method_label"] == "t=999"]
    ddim = sub[sub["method_label"] == "DDIM-100"]
    if len(t999) and len(ddim):
        print(
            f"  [{variant_short}] t=999 MSE={t999['latent_mse_mean'].iloc[0]:.4g} "
            f"PSNR={t999['PSNR_mean'].iloc[0]:.2f} | DDIM-100 MSE={ddim['latent_mse_mean'].iloc[0]:.4g} "
            f"PSNR={ddim['PSNR_mean'].iloc[0]:.2f}"
        )




## Parte III — Síntesis E1 → E2 y exportación



In [ ]:

import json

summary_df = build_e1_e2_summary_table(E1_KEYS, E2_COMPARISON)
display(summary_df.round(3))

out_dir = REPORT_EXPORT_DIR
out_dir.mkdir(parents=True, exist_ok=True)

e1_cls_all.to_csv(out_dir / "e1_classification_comparison.csv", index=False)
e2_cls_all.to_csv(out_dir / "e2_classification_comparison.csv", index=False)
e2_recon_all.to_csv(out_dir / "e2_reconstruction_test.csv", index=False)
summary_df.to_csv(out_dir / "e1_e2_transversal_summary.csv", index=False)
e2_ckpt_df.to_csv(out_dir / "e2_training_checkpoint_summary.csv", index=False)
selected.to_csv(out_dir / "selected_visualization_subjects.csv", index=False)
link_df.to_csv(out_dir / "e1_e2_link_table.csv", index=False)
e1_config_df.to_csv(out_dir / "e1_run_config_summary.csv", index=False)

if "diag_df" in globals() and isinstance(diag_df, pd.DataFrame) and len(diag_df):
    diag_df.to_csv(out_dir / "sampling_diagnosis_per_subject.csv", index=False)
if "summary" in globals() and isinstance(summary, pd.DataFrame) and len(summary):
    summary.to_csv(out_dir / "sampling_diagnosis_summary.csv", index=False)

print(f"Exportado en {out_dir}")
print("\n--- Resumen rápido ---")
for _, row in summary_df.iterrows():
    print(
        f"  [{row['e2_short']}] E1 {row['e1_feature_eval']} bal.acc={row['e1_balanced_accuracy']:.3f} "
        f"→ E2 bal.acc={row['e2_balanced_accuracy']:.3f}, PSNR={row['e2_test_psnr_mean']:.2f}"
    )


